In [17]:
import os
from pathlib import Path
from dotenv import load_dotenv

import cv2
import joblib
import numpy as np
from skimage.feature import hog

# =====================================
# Load Model
# =====================================

load_dotenv()

PROJECT_DIR = Path(os.getenv("PROJECT_DIR"))
MODEL_DIR = PROJECT_DIR / os.getenv("MODEL_DIR")
MODEL_PATH = MODEL_DIR / os.getenv("MODEL_PATH")

model = joblib.load(MODEL_PATH)

# =====================================
# Parameters
# =====================================

WINDOW_SIZE = 128
PIXELS_PER_CELL = 16
CELLS_PER_BLOCK = 2
ORIENTATIONS = 10
CELL_STEP = 2

SCALE_FACTOR = 1.25
CONFIDENCE_THRESHOLD = 0.76
# =====================================
# Image Pyramid
# =====================================

def pyramid(image,
            scale=SCALE_FACTOR,
            min_size=(WINDOW_SIZE, WINDOW_SIZE)):

    yield image

    while True:

        w = int(image.shape[1] / scale)
        h = int(image.shape[0] / scale)

        image = cv2.resize(image, (w, h))

        if image.shape[0] < min_size[1] or image.shape[1] < min_size[0]:
            break

        yield image

# =====================================
# Sliding Window Predictor
# =====================================

def predictor(gray):

    height, width = gray.shape

    hog_features = hog(
        gray,
        orientations=ORIENTATIONS,
        pixels_per_cell=(PIXELS_PER_CELL, PIXELS_PER_CELL),
        cells_per_block=(CELLS_PER_BLOCK, CELLS_PER_BLOCK),
        transform_sqrt=True,
        feature_vector=False
    )

    n_blocks_y = hog_features.shape[0]
    n_blocks_x = hog_features.shape[1]

    blocks_in_window = (WINDOW_SIZE // PIXELS_PER_CELL) - 1

    steps_x = (n_blocks_x - blocks_in_window) // CELL_STEP
    steps_y = (n_blocks_y - blocks_in_window) // CELL_STEP

    rectangles = []

    for xb in range(steps_x):

        for yb in range(steps_y):

            x_block = xb * CELL_STEP
            y_block = yb * CELL_STEP

            hog_sample = hog_features[
                y_block:y_block + blocks_in_window,
                x_block:x_block + blocks_in_window
            ].ravel()

            score = model.decision_function(
                hog_sample.reshape(1, -1)
            )[0]

            if score > CONFIDENCE_THRESHOLD:

                x = x_block * PIXELS_PER_CELL
                y = y_block * PIXELS_PER_CELL

                rectangles.append(
                    (
                        x,
                        y,
                        x + WINDOW_SIZE,
                        y + WINDOW_SIZE,
                        score
                    )
                )

    return rectangles

# =====================================
# Multi-scale Detection
# =====================================

def detect_faces(gray):

    detections = []

    scale = 1.0

    for resized in pyramid(gray):

        rects = predictor(resized)

        for (x1, y1, x2, y2, score) in rects:

            detections.append(
                (
                    int(x1 * scale),
                    int(y1 * scale),
                    int(x2 * scale),
                    int(y2 * scale),
                    score
                )
            )

        scale *= SCALE_FACTOR

    return detections

# =====================================q
# Draw Rectangles
# =====================================
def draw_boxes(frame, detections):

    image = frame.copy()

    for (x1, y1, x2, y2, score) in detections:

        cv2.rectangle(
            image,
            (x1, y1),
            (x2, y2),
            (0,255,0),
            2
        )

        cv2.putText(
            image,
            f"{score:.2f}",
            (x1, y1-5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0,255,0),
            1
        )

    return image
def apply_nms(detections,
              score_threshold=0.7,
              nms_threshold=0.3):

    if len(detections) == 0:
        return []

    boxes = []
    scores = []

    for (x1, y1, x2, y2, score) in detections:

        boxes.append([
            int(x1),
            int(y1),
            int(x2 - x1),
            int(y2 - y1)
        ])

        scores.append(float(score))

    indices = cv2.dnn.NMSBoxes(
        boxes,
        scores,
        score_threshold,
        nms_threshold
    )

    final_boxes = []

    if len(indices) == 0:
        return final_boxes

    # Compatible with all OpenCV versions
    for idx in indices:

        if isinstance(idx, (list, tuple, np.ndarray)):
            idx = idx[0]

        final_boxes.append(detections[int(idx)])

    return final_boxes

# =====================================
# Webcam
# =====================================

cap = cv2.VideoCapture(1)

while True:

    ret, frame = cap.read()

    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    detections = detect_faces(gray)

    detections = apply_nms(
        detections,
        score_threshold=0.7,
        nms_threshold=0.3
    )

    output = draw_boxes(frame, detections)

    cv2.imshow("HOG + SVM Face Detector", output)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()